# **Data Pipeline**

## **Data Source:**

[MAESTRO](https://magenta.withgoogle.com/datasets/maestro#v300), is a large-scale dataset of classical piano performances with paired audio and MIDI recordings. It was released by the Magenta team and is widely used in music information retrieval and generative music research as a benchmark. The dataset contains roughly 200 hours of recordings from the International Piano-e-Competition, and each piece is provided with useful metadata such as composer, title, and performance year. The audio is high quality, typically 44.1–48 kHz stereo, while the MIDI provides structured symbolic information such as note onsets, offsets, velocities, and pedal events.

---

In [2]:
# import packages
import os
from pathlib import Path
import yaml

# data handling
import pandas as pd
import numpy as np

# audio processing
import torch
import torchaudio

# plotting for inspection
import matplotlib.pyplot as plt

In this pipeline, `data_configs.yaml` within the configs folder, is used to simplify configuration management and centralise all parameter settings.

In [15]:
# load in yaml file 
yaml_path = Path("./configs/data_configs.yaml")

# load config
with open(yaml_path, "r") as f:
    audio_cfg = yaml.safe_load(f)

print(audio_cfg)

{'project': {'name': 'maestro dataphase', 'seed': 42}, 'paths': {'raw_data_dir': './raw/maestro-v3.0.0-midi.zip', 'metadata_csv': './raw/maestro-v3.0.0.csv', 'processed_dir': './processed', 'clips_dir': './processed/clips', 'specs_dir': './processed/spectrograms', 'manifests_dir': './processed/manifests'}, 'audio': {'sample_rate': 16000, 'mono': False, 'normalise': True, 'clip_duration_seconds': 30.0}, 'spectrogram': {'n_fft': 1024, 'hop_length': 256, 'win_length': 1024, 'n_mels': 128, 'f_min': 30, 'f_max': 8000, 'power': 2.0}, 'masking': {'strategy': 'random_safe', 'gap_seconds_options': [0.5, 1.0, 2.0, 5.0], 'safe_margin_ratio': 0.2, 'fill_value': 0.0}, 'split': {'use_official_maestro_split': True, 'train_name': 'train', 'val_name': 'validation', 'test_name': 'test'}, 'save': {'save_waveform_clips': True, 'save_mel_spectrograms': True, 'save_manifests': True, 'audio_format': 'pt', 'spectrogram_format': 'pt'}}


In [16]:
# creating directories listed in yaml configs
dirs_to_create = [
    audio_cfg["paths"]["processed_dir"],
    audio_cfg["paths"]["clips_dir"],
    audio_cfg["paths"]["specs_dir"],
    audio_cfg["paths"]["manifests_dir"],
    ]
for folder in dirs_to_create:
    Path(folder).mkdir(parents=True, exist_ok=True)

In [ ]:
# loading metadata
metadata_path = Path(audio_cfg["paths"]["metadata_csv"])
metadata_df = pd.read_csv(metadata_path)

print("\n MAETSRO metadata:")
display(metadata_df.head())
print(metadata_df.columns.tolist())
# check split distribution
print(metadata_df["split"].value_counts())


 MAETSRO metadata:


,canonical_composer,canonical_title,split,year,midi_filename,audio_filename,duration
0,Alban Berg,Sonata Op. 1,train,2018,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...,698.661160
1,Alban Berg,Sonata Op. 1,train,2008,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...,759.518471
2,Alban Berg,Sonata Op. 1,train,2017,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...,464.649433
3,Alexander Scriabin,"24 Preludes Op. 11, No. 13-24",train,2004,2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...,2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...,872.640588
4,Alexander Scriabin,"3 Etudes, Op. 65",validation,2006,2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...,2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...,397.857508


['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']
split
train         962
test          177
validation    137
Name: count, dtype: int64


In [22]:
# split metadata
train_name = audio_cfg["split"]["train_name"]
val_name = audio_cfg["split"]["val_name"]
test_name = audio_cfg["split"]["test_name"]

meta_train = metadata_df[metadata_df["split"] == train_name].copy()
meta_val = metadata_df[metadata_df["split"] == val_name].copy()
meta_test = metadata_df[metadata_df["split"] == test_name].copy()

print("Train files:", len(meta_train))
print("Validation files:", len(meta_val))
print("Test files:", len(meta_test))

Train files: 962
Validation files: 137
Test files: 177


In [24]:
# using the meta data to select one training file to inspect
eg_01 = meta_train.iloc[0]
audio_rel_path = eg_01["audio_filename"]
audio_full_path = Path(audio_cfg["paths"]["raw_data_dir"]) / audio_rel_path

In [25]:
print(audio_full_path)

raw/maestro-v3.0.0-midi.zip/2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.wav
